# Utility-policy sensitivity and CV reliability

This notebook analyses one completed chronological-CV experiment. It separates (i) the normative choice of the lead-time utility curve from (ii) the empirical reliability of the resulting model comparison. In particular, it sweeps the utility at the one-hour minimum lead time and the curve's power/shape, re-selects each alert threshold using the saved validation period only, and evaluates the corresponding held-out test folds.

**Important.** Shape sensitivity requires `cv/operational_scores.parquet` and `cv/realised_depeg_context.parquet`, written by the current `cv_model_comparison.py`. Older experiments without those files can still be summarised for fold counts and their already-recorded CIs, but cannot be honestly re-scored for a new utility curve.

In [ ]:
from pathlib import Path
import json
import sys
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

# Run from this directory, or from the repository root.
PROJECT_DIR = Path.cwd().resolve()
if not (PROJECT_DIR / 'early_warning_evaluation.py').exists():
    PROJECT_DIR = PROJECT_DIR / '8. Early-Warning Model'
if not (PROJECT_DIR / 'early_warning_evaluation.py').exists():
    raise FileNotFoundError('Start the notebook in 8. Early-Warning Model or the repository root.')
sys.path.insert(0, str(PROJECT_DIR))
from early_warning_evaluation import (
    choose_threshold_by_utility, evaluate_early_warning, event_block_bootstrap_ci,
)

# ---- User configuration ---------------------------------------------------
EXPERIMENT_NAME = 'cv_model_comparison_YYYY-MM-DD'  # replace with one completed experiment
LOG_DIR = PROJECT_DIR / 'lightning_logs'
FOCUS_DEPEG_THRESHOLD_BPS = None  # e.g. 15.0; None analyses every available definition
FALSE_ALERT_BUDGET = 2.0          # episodes/month; threshold selection constraint
FALSE_ALERT_COST = 0.05           # penalty in the utility score
MIN_LEAD_HOURS = 1.0
MAX_LEAD_HOURS = 24.0
TARGET_LEAD_HOURS = 24.0
ALERT_COOLDOWN_HOURS = 24.0
N_BOOTSTRAP = 2_000               # paired event/calendar-block bootstrap draws
OUTPUT_DIR = LOG_DIR / EXPERIMENT_NAME / 'utility_reliability_analysis'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# The baseline reproduces the current policy. Power < 1 is concave (more
# value for short, valid warnings); power > 1 is convex.
SCENARIOS = pd.DataFrame([
    {'scenario': 'baseline: linear, u(1h)=0.10', 'min_lead_utility': 0.10, 'utility_power': 1.00},
    {'scenario': 'low floor: linear, u(1h)=0.00', 'min_lead_utility': 0.00, 'utility_power': 1.00},
    {'scenario': 'high floor: linear, u(1h)=0.25', 'min_lead_utility': 0.25, 'utility_power': 1.00},
    {'scenario': 'concave: u(1h)=0.10, p=0.5', 'min_lead_utility': 0.10, 'utility_power': 0.50},
    {'scenario': 'convex: u(1h)=0.10, p=2.0', 'min_lead_utility': 0.10, 'utility_power': 2.00},
])
assert 0 <= SCENARIOS.min_lead_utility.min() <= SCENARIOS.min_lead_utility.max() <= 1
assert (SCENARIOS.utility_power > 0).all()


## 1. Utility functions being compared

For a valid lead time $l \in [m,T]$, the notebook uses $u(l)=u_m+(1-u_m)((l-m)/(T-m))^p$. Here $m$ is the minimum lead, $u_m$ is the utility exactly at that minimum, and $p$ controls shape. Warnings before $m$ receive zero; warnings at or beyond $T$ receive one.

In [ ]:
def utility_curve(lead, min_lead, target_lead, min_utility, power):
    lead = np.asarray(lead, dtype=float)
    progress = np.clip((lead - min_lead) / (target_lead - min_lead), 0, 1)
    value = min_utility + (1 - min_utility) * progress ** power
    return np.where(lead < min_lead, 0.0, value)

hours = np.linspace(0, MAX_LEAD_HOURS, 400)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2), sharey=True)
for _, s in SCENARIOS.iterrows():
    y = utility_curve(hours, MIN_LEAD_HOURS, TARGET_LEAD_HOURS, s.min_lead_utility, s.utility_power)
    axes[0].plot(hours, y, label=s.scenario)
    axes[1].plot(hours, y, label=s.scenario)
for ax in axes:
    ax.axvline(MIN_LEAD_HOURS, color='0.5', ls='--', lw=1)
    ax.axvline(TARGET_LEAD_HOURS, color='0.5', ls=':', lw=1)
    ax.set(xlabel='Hours before realised depeg', ylabel='Event utility', ylim=(-0.03, 1.05))
    ax.grid(alpha=.25)
axes[0].set_title('All proposed lead-time utility curves')
axes[1].set_title('Same curves: minimum lead is one hour')
axes[1].legend(loc='lower right', fontsize=8)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / 'utility_curve_scenarios.png', dpi=320, bbox_inches='tight')
plt.show()
display(SCENARIOS)


In [ ]:
# ---- Discover the latest CV run for each (model, alpha, depeg threshold) --
EXPERIMENT_DIR = LOG_DIR / EXPERIMENT_NAME
if not EXPERIMENT_DIR.exists():
    available = sorted(p.name for p in LOG_DIR.glob('cv_model_comparison*') if p.is_dir())
    raise FileNotFoundError(f'{EXPERIMENT_DIR} does not exist. Available examples: {available[-8:]}')

def read_json(path):
    try:
        return json.loads(path.read_text())
    except (OSError, json.JSONDecodeError):
        return {}

def discover_runs(experiment_dir):
    found = []
    for metrics_path in experiment_dir.glob('*/artifacts/cv/fold_metrics.csv'):
        run_dir = metrics_path.parents[2]
        frame = pd.read_csv(metrics_path)
        if frame.empty:
            continue
        hparams = read_json(run_dir / 'hparams.json')
        first = frame.iloc[0]
        model = str(first.get('model_name', hparams.get('model_name', run_dir.name)))
        alpha = float(first.get('alpha', hparams.get('alpha', np.nan)))
        threshold = float(first.get('target_threshold', hparams.get('target_threshold', np.nan)))
        found.append({
            'run_dir': run_dir, 'metrics_path': metrics_path, 'hparams': hparams,
            'model_name': model, 'alpha': alpha, 'target_threshold': threshold,
            'mtime': metrics_path.stat().st_mtime, 'folds': frame,
        })
    newest = {}
    for run in found:
        key = (run['model_name'], run['alpha'], run['target_threshold'])
        if key not in newest or run['mtime'] > newest[key]['mtime']:
            newest[key] = run
    return list(newest.values())

RUNS = discover_runs(EXPERIMENT_DIR)
if FOCUS_DEPEG_THRESHOLD_BPS is not None:
    RUNS = [r for r in RUNS if np.isclose(r['target_threshold'], FOCUS_DEPEG_THRESHOLD_BPS)]
if not RUNS:
    raise RuntimeError('No CV fold reports matched the requested realised-depeg threshold.')

folds = pd.concat([r['folds'].assign(run_dir=str(r['run_dir'])) for r in RUNS], ignore_index=True)
display(Markdown(f'Loaded **{len(RUNS)}** latest model/alpha specifications and **{len(folds)}** outer-fold rows.'))
display(folds[['model_name', 'alpha', 'target_threshold', 'fold', 'test_start', 'test_end']].head())


## 2. Reliability before re-scoring

The table reports boundary-complete realised-event **fold contributions** that actually enter each utility calculation, rather than positive label rows. Because chronological test windows overlap, the same realised event can contribute to more than one fold; this count is an effective evidence count, not a count of unique market episodes. The stored event/calendar-block bootstrap CIs are shown when present.

In [ ]:
required = {'fold_event_utility_score', 'fold_timely_event_recall', 'fold_false_alerts_per_month'}
missing = required - set(folds.columns)
if missing:
    raise ValueError(f'This experiment predates operational utility reporting: missing {sorted(missing)}')

group_cols = ['target_threshold', 'model_name', 'alpha']
def ci_range(g, metric):
    lo, hi = f'{metric}_ci_lower', f'{metric}_ci_upper'
    if lo in g and hi in g:
        return pd.Series({'ci_low': pd.to_numeric(g[lo], errors='coerce').mean(),
                          'ci_high': pd.to_numeric(g[hi], errors='coerce').mean()})
    return pd.Series({'ci_low': np.nan, 'ci_high': np.nan})

reliability = (folds.groupby(group_cols, dropna=False)
    .agg(n_outer_folds=('fold', 'nunique'),
         event_fold_contributions=('fold_n_events', 'sum') if 'fold_n_events' in folds else ('fold', 'size'),
         detected_events=('fold_events_detected', 'sum') if 'fold_events_detected' in folds else ('fold', 'size'),
         mean_utility=('fold_event_utility_score', 'mean'),
         sd_utility=('fold_event_utility_score', 'std'),
         mean_recall=('fold_timely_event_recall', 'mean'),
         mean_false_alerts=('fold_false_alerts_per_month', 'mean'))
    .reset_index())
utility_ci = (folds.groupby(group_cols, dropna=False)
    .apply(lambda g: ci_range(g, 'event_utility_score'))
    .reset_index())
reliability = reliability.merge(utility_ci, on=group_cols, how='left').sort_values(['target_threshold', 'mean_utility'], ascending=[True, False])
reliability.to_csv(OUTPUT_DIR / 'existing_cv_reliability_summary.csv', index=False)
display(reliability)

fig, ax = plt.subplots(figsize=(10, 4.5))
for (threshold, model, alpha), g in folds.groupby(group_cols):
    label = f'{threshold:g}bps | {model}, α={alpha:g}'
    ax.plot(g['fold'], g['fold_event_utility_score'], marker='o', alpha=.8, label=label)
ax.axhline(0, color='0.4', lw=.8, ls='--')
ax.set(xlabel='Chronological outer fold', ylabel='Held-out event utility', title='Fold-to-fold utility stability')
ax.grid(alpha=.25); ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'existing_fold_utility_stability.png', dpi=320, bbox_inches='tight'); plt.show()


## 3. Re-select thresholds under every utility scenario

This is the central sensitivity analysis. For every model/alpha/fold/scenario, the threshold is selected on the saved validation scores subject to the same false-alert budget, then evaluated once on its paired test fold. Thus the notebook does not evaluate a new utility curve at a threshold that was tuned for an old curve.

In [ ]:
def policy_for(run, scenario):
    hp = run['hparams']
    return {
        'timestamp_col': 'timestamp', 'depeg_col': 'depeg_bps',
        'threshold_bps': float(run['target_threshold']),
        'depeg_side': hp.get('depeg_side', 'both'),
        'dynamic_threshold': bool(hp.get('dynamic_threshold', False)),
        'min_lead_hours': MIN_LEAD_HOURS, 'max_lead_hours': MAX_LEAD_HOURS,
        'target_lead_hours': TARGET_LEAD_HOURS,
        'min_lead_utility': float(scenario.min_lead_utility),
        'utility_power': float(scenario.utility_power),
        'cooldown_hours': ALERT_COOLDOWN_HOURS, 'false_alert_cost': FALSE_ALERT_COST,
    }

def rescore(run, scenario):
    cv_dir = run['run_dir'] / 'artifacts' / 'cv'
    scores_path, context_path = cv_dir / 'operational_scores.parquet', cv_dir / 'realised_depeg_context.parquet'
    if not scores_path.exists() or not context_path.exists():
        return None, None
    scores, context = pd.read_parquet(scores_path), pd.read_parquet(context_path)
    scores['timestamp'] = pd.to_datetime(scores['timestamp'], utc=True)
    context['timestamp'] = pd.to_datetime(context['timestamp'], utc=True)
    rows, draws_by_fold = [], {}
    kwargs = policy_for(run, scenario)
    for fold in sorted(scores.fold.unique()):
        validation = scores[(scores.fold == fold) & (scores.split == 'validation')].sort_values('timestamp')
        test = scores[(scores.fold == fold) & (scores.split == 'test')].sort_values('timestamp')
        if validation.empty or test.empty:
            continue
        val_context = context[context.timestamp <= validation.timestamp.max()]
        test_context = context[context.timestamp <= test.timestamp.max()]
        threshold, _ = choose_threshold_by_utility(
            validation, validation.probability, FALSE_ALERT_BUDGET, 201,
            event_context_frame=val_context, **kwargs)
        metrics, inputs = evaluate_early_warning(
            test, test.probability, threshold, event_context_frame=test_context, **kwargs)
        _, draws = event_block_bootstrap_ci(
            inputs, false_alert_cost=FALSE_ALERT_COST, block_hours=24 * 7,
            n_bootstrap=N_BOOTSTRAP, random_state=50_000 + int(fold))
        rows.append({**metrics, 'fold': int(fold), 'model_name': run['model_name'],
                     'alpha': run['alpha'], 'target_threshold': run['target_threshold'],
                     'scenario': scenario.scenario, 'min_lead_utility': scenario.min_lead_utility,
                     'utility_power': scenario.utility_power})
        draws_by_fold[int(fold)] = draws
    return pd.DataFrame(rows), draws_by_fold

rescored, all_draws, unavailable = [], {}, []
for _, scenario in SCENARIOS.iterrows():
    for run in RUNS:
        result, draws = rescore(run, scenario)
        key = (scenario.scenario, run['target_threshold'], run['model_name'], run['alpha'])
        if result is None:
            unavailable.append(str(run['run_dir']))
        else:
            rescored.append(result); all_draws[key] = draws
if unavailable:
    warnings.warn('Shape re-scoring unavailable for older runs. Re-run CV to create operational score artifacts:\n' + '\n'.join(sorted(set(unavailable))))
if not rescored:
    raise RuntimeError('No saved operational score paths found; only Section 2 is available for this experiment.')
rescored = pd.concat(rescored, ignore_index=True)
rescored.to_csv(OUTPUT_DIR / 'utility_shape_rescored_folds.csv', index=False)
display(rescored.head())


In [ ]:
# Aggregate outer folds and form event/block bootstrap CIs. The same seed is
# used within a fold across models, enabling paired descriptive comparisons.
spec_cols = ['scenario', 'target_threshold', 'model_name', 'alpha']
def pooled_ci(key, metric):
    arrays = [x[metric] for x in all_draws[key].values() if metric in x]
    if not arrays:
        return (np.nan, np.nan)
    pooled = np.nanmean(np.vstack(arrays), axis=0)
    pooled = pooled[np.isfinite(pooled)]
    return (np.quantile(pooled, .025), np.quantile(pooled, .975)) if len(pooled) else (np.nan, np.nan)

summary_rows = []
for key, g in rescored.groupby(spec_cols, dropna=False):
    draw_key = tuple(key)
    row = dict(zip(spec_cols, key))
    row['n_outer_folds'] = g.fold.nunique()
    row['total_events'] = g.n_events.sum()
    for metric in ['event_utility_score', 'event_utility', 'timely_event_recall', 'false_alerts_per_month', 'median_lead_hours']:
        row[f'mean_{metric}'] = g[metric].mean()
        row[f'{metric}_ci_low'], row[f'{metric}_ci_high'] = pooled_ci(draw_key, metric)
    summary_rows.append(row)
utility_summary = pd.DataFrame(summary_rows).sort_values(['scenario', 'target_threshold', 'mean_event_utility_score'], ascending=[True, True, False])
utility_summary.to_csv(OUTPUT_DIR / 'utility_shape_summary_with_event_block_cis.csv', index=False)
display(utility_summary)

fig, ax = plt.subplots(figsize=(10, 5))
for (threshold, model, alpha), g in utility_summary.groupby(['target_threshold', 'model_name', 'alpha']):
    g = g.set_index('scenario').reindex(SCENARIOS.scenario).reset_index()
    ax.plot(g.scenario, g.mean_event_utility_score, marker='o', label=f'{threshold:g}bps | {model}, α={alpha:g}')
ax.axhline(0, color='0.4', lw=.8, ls='--')
ax.set(ylabel='Mean held-out event utility', title='Utility-policy sensitivity after validation-only threshold selection')
ax.tick_params(axis='x', rotation=30); ax.grid(axis='y', alpha=.25)
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'utility_shape_sensitivity.png', dpi=320, bbox_inches='tight'); plt.show()


## 4. Is the selected model credibly better?

For each utility scenario and realised-depeg definition, the table compares every specification to the best mean-utility specification using paired bootstrap draws within the same chronological folds. A 95% interval excluding zero is reported as interval evidence of a difference. Because outer test windows overlap, do **not** describe this as an independent-fold p-value or a confirmatory significance test; report it as a paired resampling sensitivity analysis.

In [ ]:
comparison_rows = []
for (scenario, threshold), g in utility_summary.groupby(['scenario', 'target_threshold']):
    winner = g.loc[g.mean_event_utility_score.idxmax()]
    winner_key = (scenario, threshold, winner.model_name, winner.alpha)
    for _, candidate in g.iterrows():
        candidate_key = (scenario, threshold, candidate.model_name, candidate.alpha)
        common = sorted(set(all_draws[winner_key]) & set(all_draws[candidate_key]))
        draw_deltas = []
        for fold in common:
            a = all_draws[winner_key][fold]['event_utility_score']
            b = all_draws[candidate_key][fold]['event_utility_score']
            draw_deltas.append(a - b)
        delta_draws = np.nanmean(np.vstack(draw_deltas), axis=0) if draw_deltas else np.array([])
        delta_draws = delta_draws[np.isfinite(delta_draws)]
        low, high = (np.quantile(delta_draws, [.025, .975]) if len(delta_draws) else (np.nan, np.nan))
        comparison_rows.append({
            'scenario': scenario, 'target_threshold': threshold,
            'winner': f"{winner.model_name} (α={winner.alpha:g})",
            'comparator': f"{candidate.model_name} (α={candidate.alpha:g})",
            'mean_utility_difference_winner_minus_comparator': winner.mean_event_utility_score - candidate.mean_event_utility_score,
            'paired_bootstrap_ci_low': low, 'paired_bootstrap_ci_high': high,
            'probability_winner_higher': float(np.mean(delta_draws > 0)) if len(delta_draws) else np.nan,
            'interval_excludes_zero': bool(low > 0 or high < 0) if np.isfinite(low) and np.isfinite(high) else False,
            'common_outer_folds': len(common),
        })
paired_comparisons = pd.DataFrame(comparison_rows)
paired_comparisons.to_csv(OUTPUT_DIR / 'paired_utility_comparisons.csv', index=False)
display(paired_comparisons)
display(Markdown('**Paper wording:** “The preferred specification had the highest mean outer-fold utility under the stated utility policy. Paired event/calendar-block bootstrap intervals quantify sensitivity to realised events and alert clustering; overlapping chronological test windows preclude interpreting these intervals as independent-fold hypothesis tests.”'))


## 5. Paper-ready figures and LaTeX tables

This final cell writes vector PDFs, 350-dpi PNGs, CSV files, and LaTeX tables into `utility_reliability_analysis/`. The selected specification is reselected separately for every utility scenario and realised-depeg threshold.

In [ ]:
PAPER_DIR = OUTPUT_DIR / 'paper_ready'
PAPER_DIR.mkdir(exist_ok=True)
BASELINE_SCENARIO = SCENARIOS.iloc[0].scenario

def save_paper_figure(fig, stem):
    fig.savefig(PAPER_DIR / f'{stem}.png', dpi=350, bbox_inches='tight')
    fig.savefig(PAPER_DIR / f'{stem}.pdf', bbox_inches='tight')
    plt.close(fig)

def ci_text(row, metric, digits=3):
    point, low, high = row[f'mean_{metric}'], row[f'{metric}_ci_low'], row[f'{metric}_ci_high']
    if pd.notna(low) and pd.notna(high):
        return f'{point:.{digits}f} [{low:.{digits}f}, {high:.{digits}f}]'
    return f'{point:.{digits}f}' if pd.notna(point) else 'NA'

# Select the highest-mean-utility specification independently at each policy.
selected = (utility_summary.sort_values('mean_event_utility_score', ascending=False)
    .groupby(['scenario', 'target_threshold'], group_keys=False).head(1).copy())
selected['selected_specification'] = selected.apply(lambda r: f"{r.model_name} ($\\alpha={r.alpha:g}$)", axis=1)
selected['event_utility_95ci'] = selected.apply(ci_text, axis=1, metric='event_utility_score')
selected['timely_recall_95ci'] = selected.apply(ci_text, axis=1, metric='timely_event_recall')
selected['false_alerts_95ci'] = selected.apply(lambda r: ci_text(r, 'false_alerts_per_month', 2), axis=1)
selected['median_lead_95ci'] = selected.apply(lambda r: ci_text(r, 'median_lead_hours', 1), axis=1)
selected_table = selected[[
    'target_threshold', 'scenario', 'selected_specification', 'n_outer_folds', 'total_events',
    'event_utility_95ci', 'timely_recall_95ci', 'false_alerts_95ci', 'median_lead_95ci'
]].rename(columns={
    'target_threshold': 'Depeg threshold (bps)', 'scenario': 'Utility policy',
    'selected_specification': 'Selected specification', 'n_outer_folds': 'Outer folds',
    'total_events': 'Event-fold contributions', 'event_utility_95ci': 'Event utility (95% CI)',
    'timely_recall_95ci': 'Timely recall (95% CI)', 'false_alerts_95ci': 'False alerts/month (95% CI)',
    'median_lead_95ci': 'Median lead hours (95% CI)'
})
selected_table.to_csv(PAPER_DIR / 'table_utility_policy_sensitivity.csv', index=False)
(PAPER_DIR / 'table_utility_policy_sensitivity.tex').write_text(
    selected_table.to_latex(index=False, escape=False, na_rep='NA', longtable=len(selected_table) > 12))

comparison_table = paired_comparisons[paired_comparisons.scenario.eq(BASELINE_SCENARIO)].copy()
comparison_table = comparison_table[comparison_table.winner.ne(comparison_table.comparator)]
comparison_table['difference_95ci'] = comparison_table.apply(
    lambda r: f"{r.mean_utility_difference_winner_minus_comparator:.3f} [{r.paired_bootstrap_ci_low:.3f}, {r.paired_bootstrap_ci_high:.3f}]", axis=1)
comparison_table['Interval excludes zero'] = comparison_table.interval_excludes_zero.map({True: 'Yes', False: 'No'})
comparison_table = comparison_table[[
    'target_threshold', 'winner', 'comparator', 'difference_95ci', 'probability_winner_higher',
    'Interval excludes zero', 'common_outer_folds'
]].rename(columns={
    'target_threshold': 'Depeg threshold (bps)', 'winner': 'Preferred specification',
    'comparator': 'Comparator', 'difference_95ci': 'Utility difference (95% CI)',
    'probability_winner_higher': 'Bootstrap P(preferred > comparator)',
    'common_outer_folds': 'Paired outer folds'
})
comparison_table.to_csv(PAPER_DIR / 'table_paired_model_comparisons.csv', index=False)
(PAPER_DIR / 'table_paired_model_comparisons.tex').write_text(
    comparison_table.to_latex(index=False, escape=False, na_rep='NA', longtable=len(comparison_table) > 12))

# Figure 1: selected model’s policy sensitivity, with event/block bootstrap CIs.
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharex=True)
for threshold, g in selected.groupby('target_threshold'):
    g = g.set_index('scenario').reindex(SCENARIOS.scenario).reset_index()
    x = np.arange(len(g))
    for ax, metric, label in [(axes[0], 'event_utility_score', 'Held-out event utility'),
                              (axes[1], 'timely_event_recall', 'Timely event recall')]:
        y = g[f'mean_{metric}'].to_numpy(float)
        lo, hi = g[f'{metric}_ci_low'].to_numpy(float), g[f'{metric}_ci_high'].to_numpy(float)
        ax.errorbar(x, y, yerr=np.vstack([np.maximum(y-lo, 0), np.maximum(hi-y, 0)]),
                    marker='o', capsize=3, label=f'{threshold:g} bps')
        ax.set_ylabel(label); ax.grid(axis='y', alpha=.25)
for ax in axes:
    ax.set_xticks(np.arange(len(SCENARIOS))); ax.set_xticklabels(SCENARIOS.scenario, rotation=28, ha='right', fontsize=8)
axes[0].axhline(0, color='0.4', lw=.8, ls='--'); axes[1].set_ylim(-.03, 1.03)
axes[0].set_title('Utility-policy sensitivity'); axes[1].set_title('Timely recall under each policy')
axes[1].legend(title='Realised-depeg threshold', frameon=False)
fig.tight_layout(); save_paper_figure(fig, 'figure_utility_policy_sensitivity')

# Figure 2: paired utility-difference forest plot at the baseline policy.
plot_df = paired_comparisons[(paired_comparisons.scenario == BASELINE_SCENARIO) &
                             (paired_comparisons.winner != paired_comparisons.comparator)].copy()
if not plot_df.empty:
    plot_df['label'] = plot_df.target_threshold.map(lambda x: f'{x:g} bps') + ': ' + plot_df.comparator
    plot_df = plot_df.sort_values('mean_utility_difference_winner_minus_comparator')
    y = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=(9, max(3.2, .45 * len(plot_df) + 1.2)))
    point = plot_df.mean_utility_difference_winner_minus_comparator.to_numpy(float)
    low, high = plot_df.paired_bootstrap_ci_low.to_numpy(float), plot_df.paired_bootstrap_ci_high.to_numpy(float)
    ax.errorbar(point, y, xerr=np.vstack([np.maximum(point-low, 0), np.maximum(high-point, 0)]),
                fmt='o', capsize=3, color='darkorange')
    ax.axvline(0, color='0.35', ls='--', lw=1)
    ax.set(yticks=y, yticklabels=plot_df.label, xlabel='Utility difference: preferred − comparator',
           title='Paired event/block-bootstrap comparison at baseline utility policy')
    ax.grid(axis='x', alpha=.25); fig.tight_layout(); save_paper_figure(fig, 'figure_paired_model_utility_differences')

display(Markdown(f'Paper-ready artifacts written to `{PAPER_DIR}`.'))
display(selected_table)
display(comparison_table)
